<a href="https://colab.research.google.com/github/Silva015/tcc-2026/blob/main/Entrada_Saida_ConvKAN%2BResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CONV KAN + RESNET ENTRADA SAIDA

In [1]:
import sys
sys.path.insert(0, '/content/torch-conv-kan')
import models.reskanet as rk

print("🔍 MODELOS DISPONÍVEIS NESTE ARQUIVO:")
for nome in dir(rk):
    # Filtra para mostrar apenas o que parece ser uma função de modelo (ignora classes internas e variáveis)
    if not nome.startswith('__') and ('res' in nome.lower() or 'kan' in nome.lower()):
        print(f" -> {nome}")

🔍 MODELOS DISPONÍVEIS NESTE ARQUIVO:
 -> FastKANBasicBlock
 -> FastKANBottleneck
 -> FastKANConv2DLayer
 -> KANBasicBlock
 -> KANBottleneck
 -> KANConv2DLayer
 -> MoEResKANet
 -> ResKANet
 -> fast_kan_conv1x1
 -> fast_kan_conv3x3
 -> fast_reskanet_18x32p
 -> kan_conv1x1
 -> kan_conv3x3
 -> moe_reskalnet_101x64p
 -> moe_reskalnet_152x64p
 -> moe_reskalnet_18x64p
 -> moe_reskalnet_50x64p
 -> reskacnet_18x32p
 -> reskagnet101
 -> reskagnet152
 -> reskagnet18
 -> reskagnet50
 -> reskagnet_101x64p
 -> reskagnet_18x32p
 -> reskagnet_34x32p
 -> reskagnet_bn50
 -> reskagnetbn_18x32p
 -> reskagnetbn_34x32p
 -> reskagnetbn_moe_18x32p
 -> reskagnetbn_moe_34x32p
 -> reskalnet_101x32p
 -> reskalnet_101x64p
 -> reskalnet_152x32p
 -> reskalnet_152x64p
 -> reskalnet_18x32p
 -> reskalnet_18x64p
 -> reskalnet_50x64p
 -> reskanet_18x32p


In [2]:
# Instalação da implementação eficiente da rede KAN e do Albumentations
!pip install git+https://github.com/Blealtan/efficient-kan -q
!pip install albumentations -q

# Clonagem dos repositórios contendo os dados e códigos base
!git clone https://gitlab.com/lisa-unb/leguwoi.git -q
!git clone https://github.com/pedrogarciafreitas/FDCPUnBGunshotDB.git -q

# 🚀 NOVO: Clonar repositório com arquiteturas ConvKAN avançadas
!git clone https://github.com/IvanDrokin/torch-conv-kan.git -q

import sys
# Adiciona a raiz do repo para os imports internos deles funcionarem
sys.path.append('/content/torch-conv-kan')

import os
from glob import glob
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
from efficient_kan import KAN
import copy
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def inspecionar_diretorio(caminho_base):
    print(f"--- RAIO-X DO DIRETÓRIO: {caminho_base} ---")
    for raiz, diretorios, arquivos in os.walk(caminho_base):
        if '.git' not in raiz:
            print(f"📂 Pasta: {raiz}")
            print(f"   -> Contém {len(arquivos)} arquivos e {len(diretorios)} subpastas.")
            if len(arquivos) > 0:
                print(f"   -> Amostras: {arquivos[:3]}\n")

inspecionar_diretorio('./FDCPUnBGunshotDB')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
fatal: destination path 'leguwoi' already exists and is not an empty directory.
fatal: destination path 'FDCPUnBGunshotDB' already exists and is not an empty directory.
fatal: destination path 'torch-conv-kan' already exists and is not an empty directory.
--- RAIO-X DO DIRETÓRIO: ./FDCPUnBGunshotDB ---
📂 Pasta: ./FDCPUnBGunshotDB
   -> Contém 3 arquivos e 3 subpastas.
   -> Amostras: ['README.md', 'CODES.pdf', 'CODIGOS.pdf']

📂 Pasta: ./FDCPUnBGunshotDB/database
   -> Contém 0 arquivos e 2 subpastas.
📂 Pasta: ./FDCPUnBGunshotDB/database/SAIDAS_EQX
   -> Contém 671 arquivos e 0 subpastas.
   -> Amostras: ['2014.1332SH17.38PTTSWATAFALSEXSN03F03_EQX.JPG', '2021.0375SH40.38PSTSWATAFALSEXSN02F04_EQX.JPG', '2014.0841SH50.38PSTSWATAFALSEXSN03F01_EQX.JPG']

📂 Pasta: ./FDCPUnBGunshotDB/database/ENTRADAS_EQX
   -> Contém 0 arquivos e 3 subpastas.
📂 Pasta: ./FDCPU

## 2. Ingestão de Dados: Construindo o GunshotDataset (PyTorch)

Esta célula define a "ponte" entre os arquivos de imagem soltos nas pastas do seu Google Drive/Colab e a rede neural. Ela cria uma classe personalizada (`GunshotDataset`) herdando o padrão do PyTorch, que ensina o sistema exatamente como procurar, ler e rotular cada fotografia.

**O que esta célula faz:**

* **Mapeamento Automático (Scraping local):** Utiliza a biblioteca `glob` para vasculhar automaticamente as subpastas em busca de arquivos `.JPG`. Isso evita que você tenha que listar o nome de cada arquivo manualmente.
* **Rotulagem Semântica (Labels):** Associa matematicamente a pasta de origem à classe do ferimento. Imagens vindas da pasta `ENTRADAS_EQX` recebem o rótulo **0**, enquanto as da pasta `SAIDAS_EQX` recebem o rótulo **1**.
* **Carregamento Sob Demanda (Lazy Loading):** Através do método `__getitem__`, as imagens não são todas carregadas na memória RAM de uma só vez (o que travaria o sistema). Em vez disso, o PyTorch abre e converte a imagem para o padrão de cores RGB apenas no exato momento em que ela for necessária para o treinamento.
* **Aplicação de Transformações:** Verifica se algum filtro/transformação (como os definidos para Treino ou Teste) foi passado e aplica essas edições na imagem instantaneamente antes de entregá-la para o modelo.

In [3]:
class GunshotDataset(Dataset):
    """
    Carrega as imagens de ferimentos de arma de fogo e atribui as labels corretas.
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        print("🔄 Mapeando imagens da base de dados...")

        # 1. Mapeamento das imagens de ENTRADA (Label 0)
        caminho_entradas = os.path.join(root_dir, 'database', 'ENTRADAS_EQX', '**', '*.JPG')
        for path in glob(caminho_entradas, recursive=True):
            self.image_paths.append(path)
            self.labels.append(0)

        # 2. Mapeamento das imagens de SAÍDA (Label 1)
        caminho_saidas = os.path.join(root_dir, 'database', 'SAIDAS_EQX', '*.JPG')
        for path in glob(caminho_saidas):
            self.image_paths.append(path)
            self.labels.append(1)

        print(f"✅ Concluído! Total de imagens encontradas: {len(self.image_paths)}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            # Albumentations requer arrays numpy
            image_np = np.array(image)
            augmented = self.transform(image=image_np)
            image = augmented['image']

        return image, label

## 3. Preparação de Dados e Separação Semântica Segura (Treino/Teste)

Esta célula é responsável por construir o pipeline de ingestão de dados para o modelo. O foco principal desta etapa é **prevenir o vazamento de dados (data leakage)**. Em vez de realizar um *split* aleatório das imagens (o que poderia colocar imagens diferentes do mesmo caso tanto no treino quanto no teste), a divisão é estritamente baseada no **ID do Caso** (identificado pelos primeiros 9 caracteres do arquivo).

**O que esta célula faz:**

* **Transformações e Data Augmentation:** Cria dois pipelines distintos. O treino recebe transformações agressivas (rotação, espelhamento, jitter de cor) para evitar *overfitting*, enquanto o teste recebe apenas as transformações essenciais (redimensionamento e normalização) para garantir uma métrica de avaliação justa.
* **Reprodutibilidade:** Trava as sementes aleatórias (`seed = 42`) para que a divisão dos dados produza sempre os mesmos resultados em execuções futuras.
* **Separação por Caso (80/20):** Mapeia quais imagens pertencem a qual caso e utiliza o `train_test_split` do *scikit-learn* diretamente na lista de casos únicos, direcionando 80% dos casos para treino e 20% para teste.
* **Criação dos DataLoaders:** Instancia os `DataLoaders` do PyTorch (com tamanho de lote de 32) prontos para iterar durante o loop de treinamento.
* **Sanity Check Automático:** Roda um script de validação de prova real no final da célula que cruza os IDs do treino e do teste. Se a interseção for zero, ele confirma o sucesso; caso contrário, dispara um alerta de vazamento de dados.

In [4]:
import random

# Transformações com Albumentations (Prevenção de Overfitting + Variabilidade Forense)
transform_treino = A.Compose([
    A.Resize(128, 128),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3), # Realce de contraste para detalhes do tecido
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3), # Simula distorção e elasticidade da pele
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2), # Simula granularidade de fotos com pouca luz
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Transformações puras para o TESTE
transform_teste = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# 1. Configuração de Sementes
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# 2. Instanciação Dupla
dataset_base_treino = GunshotDataset(root_dir='./FDCPUnBGunshotDB', transform=transform_treino)
dataset_base_teste  = GunshotDataset(root_dir='./FDCPUnBGunshotDB', transform=transform_teste)

# 3. Separação Semântica (Por Caso)
case_to_indices = {}
for idx, path in enumerate(dataset_base_treino.image_paths):
    filename = os.path.basename(path)
    case_id = filename[:9]
    if case_id not in case_to_indices:
        case_to_indices[case_id] = []
    case_to_indices[case_id].append(idx)

unique_cases = list(case_to_indices.keys())

cases_treino, cases_teste = train_test_split(unique_cases, test_size=0.2, random_state=RANDOM_SEED)

# 4. Mapeamento de índices
indices_treino = []
for case in cases_treino:
    indices_treino.extend(case_to_indices[case])

indices_teste = []
for case in cases_teste:
    indices_teste.extend(case_to_indices[case])

# 5. Subsets e DataLoaders
dataset_treino = Subset(dataset_base_treino, indices_treino)
dataset_teste  = Subset(dataset_base_teste, indices_teste)

trainloader = DataLoader(dataset_treino, batch_size=8, shuffle=True)
testloader  = DataLoader(dataset_teste, batch_size=8, shuffle=False)

print("\n" + "="*50)
print("🚀 SANITY CHECK DE VAZAMENTO DE DADOS (CÓDIGO)")
print("="*50)

train_cases_check = set([os.path.basename(dataset_base_treino.image_paths[i])[:9] for i in indices_treino])
test_cases_check  = set([os.path.basename(dataset_base_teste.image_paths[i])[:9] for i in indices_teste])
intersecao = train_cases_check.intersection(test_cases_check)

print(f"📊 Total de Casos Únicos: {len(unique_cases)}")
print(f"   -> Casos no Treino: {len(train_cases_check)} casos ({len(dataset_treino)} imagens)")
print(f"   -> Casos no Teste:  {len(test_cases_check)} casos ({len(dataset_teste)} imagens)")

if len(intersecao) == 0:
    print("\n✅ SUCESSO! Zero interseção semântica identificada pelo código.")
else:
    print(f"\n❌ ALERTA DE VAZAMENTO! {len(intersecao)} casos vazados: {intersecao}")
print("="*50)

🔄 Mapeando imagens da base de dados...
✅ Concluído! Total de imagens encontradas: 2554
🔄 Mapeando imagens da base de dados...
✅ Concluído! Total de imagens encontradas: 2554

🚀 SANITY CHECK DE VAZAMENTO DE DADOS (CÓDIGO)
📊 Total de Casos Únicos: 486
   -> Casos no Treino: 388 casos (2038 imagens)
   -> Casos no Teste:  98 casos (516 imagens)

✅ SUCESSO! Zero interseção semântica identificada pelo código.


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_24865/1330695307.py:10: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3), # Simula distorção e elasticidade da pele
/tmp/ipykernel_24865/1330695307.py:11: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2), # Simula granularidade de fotos com pouca luz


## 4. Arquitetura do Modelo: TransferKAN (ResNet152 + KAN)

Esta célula define a "espinha dorsal" da sua rede neural. Ela constrói uma arquitetura híbrida poderosa que mescla a robustez consolidada do *Transfer Learning* tradicional com a inovação matemática das Redes de Kolmogorov-Arnold (KAN).

**O que esta célula faz:**

* **Extrator de Características (Backbone):** Importa uma `ResNet152` com pesos pré-treinados (a versão mais profunda e poderosa da família ResNet). O código remove de forma inteligente a última camada original (o classificador antigo), aproveitando apenas a capacidade excepcional dessa rede de extrair padrões visuais complexos das imagens.
* **Regularização Agressiva (Dropout):** Implementa um `Dropout` de 50%. Durante o treinamento, metade das conexões é "desligada" aleatoriamente a cada passo. Isso impede que a rede decore as imagens de treino (memorização) e a força a aprender as características reais e generalizáveis que diferenciam os ferimentos.
* **O Diferencial (Classificador KAN):** Em vez de usar as tradicionais camadas densas (MLP), o código acopla o inovador módulo `KAN` ao final da rede. O KAN recebe o super-vetor de 2048 características da ResNet, aplica suas funções de ativação não-lineares parametrizadas em uma camada oculta de 128 dimensões, e afunila a informação para exatamente **2 saídas** (neste caso, a predição entre ferimento de entrada ou saída).
* **Fluxo de Processamento (Forward Pass):** Define o trajeto exato dos dados: a imagem entra, é mapeada espacialmente pela ResNet, achatada em um vetor 1D (*flatten*), purificada pelo *dropout* e, por fim, classificada pelo *KAN*.
* **Alocação de Hardware:** Instancia o modelo pronto e o aloca automaticamente na memória correta (`device`), garantindo que ele rode na GPU caso esteja disponível.

In [5]:
import torch.nn as nn
import sys

# Garante que a pasta do github tenha prioridade absoluta
sys.path.insert(0, '/content/torch-conv-kan')

# Importa o modelo KAN Puro
from models.reskanet import reskanet_18x32p

class ConvKANWithResNet_Gunshot(nn.Module):
    def __init__(self):
        super(ConvKANWithResNet_Gunshot, self).__init__()

        # 🟢 CORREÇÃO AQUI: Adicionamos input_channels=3 para imagens RGB
        self.backbone = reskanet_18x32p(input_channels=3, num_classes=2)

    def forward(self, x):
        return self.backbone(x)

model_gunshot = ConvKANWithResNet_Gunshot().to(device)
print(f"🚀 Modelo ConvKANWithResNet (Puro KAN) inicializado em: {device.type.upper()}")

🚀 Modelo ConvKANWithResNet (Puro KAN) inicializado em: CUDA


## 5. Loop de Treinamento e Fine-Tuning do Modelo

Esta célula executa o coração do projeto: o loop de treinamento e avaliação da rede neural. Aqui, o modelo ajusta seus pesos iterativamente ao longo de 60 épocas para aprender a classificar as imagens corretamente, utilizando técnicas avançadas para garantir a melhor convergência possível.

**O que esta célula faz:**

* **Otimização com AdamW:** Utiliza o otimizador `AdamW`, que é o padrão-ouro atual para *Fine-Tuning* em visão computacional. Ele aplica uma taxa de aprendizado controlada e uma penalidade de regularização (*weight decay*) para dificultar que o modelo decore os dados (overfitting).
* **Agendador de Taxa de Aprendizado (Scheduler):** Implementa o `CosineAnnealingLR`. Ele reduz a taxa de aprendizado de forma suave seguindo o formato de uma curva de cosseno. Isso permite que o modelo dê passos maiores no início do treino e passos bem pequenos no final, ajudando a "pousar" suavemente no melhor resultado.
* **Alternância Treino/Teste:** Para cada época, o código primeiro treina a rede (`model.train()`, onde ocorre o *backpropagation*) e, em seguida, avalia o desempenho nos dados separados (`model.eval()`, com cálculo de gradientes desligado para economizar memória).
* **Salvamento de Checkpoint Automático:** Funciona como um sistema de "Save State" automático. O código monitora a acurácia de teste e, sempre que o modelo quebra o próprio recorde (🏆), salva os pesos no arquivo `melhor_modelo_tiros.pth`. Isso garante que você termine com a melhor versão possível, mesmo que o desempenho piore nas épocas finais.

In [6]:
# ====================================================
# 1. CÁLCULO DINÂMICO DE PESOS PARA O LOSS
# ====================================================
labels_treino = [dataset_base_treino.labels[i] for i in indices_treino]
num_entradas = labels_treino.count(0)
num_saidas = labels_treino.count(1)
total_amostras = len(labels_treino)

peso_entrada = total_amostras / (2.0 * num_entradas)
peso_saida = total_amostras / (2.0 * num_saidas)

pesos_classes = torch.tensor([peso_entrada, peso_saida], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos_classes)

print(f"⚖️ Pesos de Classe Aplicados - Entrada (0): {peso_entrada:.2f} | Saída (1): {peso_saida:.2f}")

# ====================================================
# 2. CONFIGURAÇÃO DE TREINAMENTO END-TO-END
# ====================================================
total_epochs = 60 # Modelos from scratch costumam precisar de mais tempo de convergência

best_acc = 0.0
best_model_wts = copy.deepcopy(model_gunshot.state_dict())

# Otimizador ajusta todos os parâmetros do zero
optimizer = torch.optim.AdamW(model_gunshot.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)

print(f"🔥 Iniciando Treinamento End-to-End da ResKANet ({total_epochs} épocas no total)...")

for epoch in range(total_epochs):
    # ================= FASE DE TREINO =================
    model_gunshot.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_gunshot(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        running_loss += loss.item()

    acc_train = 100 * correct_train / total_train

    # ================= FASE DE TESTE =================
    model_gunshot.eval()
    correct_test, total_test = 0, 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_gunshot(images)

            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    acc_test = 100 * correct_test / total_test
    scheduler.step()

    # ================= CHECKPOINT =================
    if acc_test > best_acc:
        best_acc = acc_test
        best_model_wts = copy.deepcopy(model_gunshot.state_dict())
        torch.save(best_model_wts, 'melhor_modelo_tiros.pth')
        indicador_recorde = "🏆 NOVO RECORDE!"
    else:
        indicador_recorde = ""

    print(f"Época {epoch+1:03d}/{total_epochs} | Treino: {acc_train:.2f}% (Loss: {running_loss/len(trainloader):.4f}) | Teste: {acc_test:.2f}% {indicador_recorde}")

print(f"\n✅ Treinamento concluído. Melhor Acurácia de Validação: {best_acc:.2f}%")

⚖️ Pesos de Classe Aplicados - Entrada (0): 0.68 | Saída (1): 1.89
🔥 Iniciando Treinamento End-to-End da ResKANet (60 épocas no total)...
Época 001/60 | Treino: 59.03% (Loss: 0.7154) | Teste: 68.99% 🏆 NOVO RECORDE!
Época 002/60 | Treino: 60.06% (Loss: 0.7056) | Teste: 45.74% 
Época 003/60 | Treino: 60.89% (Loss: 0.7022) | Teste: 39.15% 
Época 004/60 | Treino: 61.09% (Loss: 0.6918) | Teste: 66.67% 
Época 005/60 | Treino: 62.71% (Loss: 0.6842) | Teste: 66.67% 
Época 006/60 | Treino: 63.74% (Loss: 0.6728) | Teste: 72.09% 🏆 NOVO RECORDE!
Época 007/60 | Treino: 65.51% (Loss: 0.6588) | Teste: 63.18% 
Época 008/60 | Treino: 64.23% (Loss: 0.6608) | Teste: 67.83% 
Época 009/60 | Treino: 65.01% (Loss: 0.6564) | Teste: 71.12% 
Época 010/60 | Treino: 65.55% (Loss: 0.6452) | Teste: 74.61% 🏆 NOVO RECORDE!
Época 011/60 | Treino: 68.84% (Loss: 0.6310) | Teste: 74.81% 🏆 NOVO RECORDE!
Época 012/60 | Treino: 69.04% (Loss: 0.6276) | Teste: 71.12% 
Época 013/60 | Treino: 68.35% (Loss: 0.6185) | Teste: 73.2

## 6. Avaliação Final e Geração de Métricas Científicas

Esta célula representa a etapa de conclusão e análise rigorosa do seu projeto. Nela, o sistema resgata a melhor versão do modelo que foi salva e o testa no conjunto de dados separado (teste). O grande diferencial aqui é a formatação dos resultados, que já são entregues no padrão exato para tabelas de publicações acadêmicas.

**O que esta célula faz:**

* **Recuperação do Melhor Modelo:** Carrega os pesos do arquivo `melhor_modelo_tiros.pth` gerado na etapa anterior, garantindo que a avaliação seja feita na versão que obteve o pico de desempenho, e não necessariamente na última época.
* **Inferência e Extração de Probabilidades:** Passa as imagens de teste pelo modelo para recolher não apenas a decisão final da classe (Entrada/Saída), mas também o grau de certeza (probabilidades via Softmax), que é fundamental para calcular a métrica AUC.
* **Cálculo Avançado de Métricas:** Vai muito além da Acurácia básica. A célula extrai as variáveis puras da Matriz de Confusão (TP, TN, FP, FN) para calcular Precisão, Recall, F1-Score, AUC e Especificidade.
* **Abordagem Multidimensional (Macro, Micro, Weighted):** Calcula as métricas usando três médias estatísticas diferentes. Isso demonstra rigor científico e prova que a avaliação não foi mascarada por um possível desbalanceamento das classes.
* **Tabela "Copy & Paste" para Artigos:** Imprime no console a "Tabela 5", um layout formatado em texto alinhado, pronto para ser transcrito para o seu artigo ou relatório científico.
* **Matriz de Confusão Visual:** Gera um gráfico de calor (*heatmap*) usando a biblioteca Seaborn, ilustrando visualmente de forma clara onde o modelo acertou e quais classes ele mais confundiu.

In [7]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("📥 Carregando o estado ótimo do modelo para geração de métricas...")
model_gunshot.load_state_dict(torch.load('melhor_modelo_tiros.pth'))
model_gunshot.eval()

y_true = []
y_pred = []
y_prob_both = []

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        outputs = model_gunshot(images)

        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_prob_both.extend(probabilities.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob_both = np.array(y_prob_both)

y_true_onehot = np.eye(2)[y_true]

# ==========================================
# EXTRAÇÃO DE VARIÁVEIS DA MATRIZ DE CONFUSÃO
# ==========================================
cm = confusion_matrix(y_true, y_pred)

FP = cm.sum(axis=0) - np.diag(cm)
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)

support_per_class = cm.sum(axis=1)

# ==========================================
# CÁLCULO DE MÉTRICAS (Fórmulas do Artigo)
# ==========================================
acc = accuracy_score(y_true, y_pred)

metrics = {}
averages = ['macro', 'micro', 'weighted']

for avg in averages:
    metrics[f'Prec_{avg}'] = precision_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'Rec_{avg}'] = recall_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'F1_{avg}'] = f1_score(y_true, y_pred, average=avg, zero_division=0)
    metrics[f'AUC_{avg}'] = roc_auc_score(y_true_onehot, y_prob_both, average=avg)

specificity_per_class = TN / (TN + FP)
metrics['Spec_macro'] = np.mean(specificity_per_class)
metrics['Spec_micro'] = TN.sum() / (TN.sum() + FP.sum())
metrics['Spec_weighted'] = np.average(specificity_per_class, weights=support_per_class)

# ==========================================
# GERAÇÃO DA TABELA DO ARTIGO (Console)
# ==========================================
print("\n" + "="*145)
print(" TABELA 5 - PERFORMANCE METRICS (FORMATO CIENTÍFICO)")
print("="*145)

header1 = f"{'Architecture':<14} {'Variant':<10} {'ACC':<7} "
header1 += f"{'Precision':<23} {'Recall':<23} {'F1-Score':<23} {'AUC':<23} {'Specificity':<23}"
print(header1)

header2 = f"{'':<33} "
for _ in range(5):
    header2 += f"{'M':<7} {'m':<7} {'W':<7}   "
print(header2)
print("-" * 145)

row = f"{'ResKANet':<14} {'Gunshot':<10} {acc:<7.3f} "
row += f"{metrics['Prec_macro']:<7.3f} {metrics['Prec_micro']:<7.3f} {metrics['Prec_weighted']:<7.3f}   "
row += f"{metrics['Rec_macro']:<7.3f} {metrics['Rec_micro']:<7.3f} {metrics['Rec_weighted']:<7.3f}   "
row += f"{metrics['F1_macro']:<7.3f} {metrics['F1_micro']:<7.3f} {metrics['F1_weighted']:<7.3f}   "
row += f"{metrics['AUC_macro']:<7.3f} {metrics['AUC_micro']:<7.3f} {metrics['AUC_weighted']:<7.3f}   "
row += f"{metrics['Spec_macro']:<7.3f} {metrics['Spec_micro']:<7.3f} {metrics['Spec_weighted']:<7.3f}"
print(row)
print("="*145 + "\n")

📥 Carregando o estado ótimo do modelo para geração de métricas...

 TABELA 5 - PERFORMANCE METRICS (FORMATO CIENTÍFICO)
Architecture   Variant    ACC     Precision               Recall                  F1-Score                AUC                     Specificity            
                                  M       m       W         M       m       W         M       m       W         M       m       W         M       m       W         
-------------------------------------------------------------------------------------------------------------------------------------------------
ResKANet       Gunshot    0.822   0.767   0.822   0.820     0.762   0.822   0.822     0.765   0.822   0.821     0.839   0.885   0.839     0.762   0.822   0.703  

